In [ ]:
import torch

input_tensor = torch.tensor([3.])

w1 = torch.tensor([2.], requires_grad=True)
w_mu = torch.tensor([8.], requires_grad=True)
w_sd = torch.tensor([9.], requires_grad=True)

enc_out = input_tensor * w1
mu = w_mu * enc_out
std = w_sd * enc_out

z = mu + std * torch.randn(1)

z.backward()

w1.grad


In [ ]:
import torch.nn as nn

In [ ]:
class VanillaAutoEncoder(nn.Module):

    def __init__(self, bottleneck_size=2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32*32),
            nn.Tanh()
        )

    def forward_enc(self, x):
        return self.encoder(x)

    def forward_dec(self, x):
        x = self.decoder(x)
        x = x.reshape(-1, 1, 32, 32)
        return x

    def forward(self, x):
        batch, channels, height, width = x.shape

        x = x.flatten(1)

        enc = self.forward_enc(x)

        dec = self.forward_dec(enc)

        return enc, dec

In [ ]:
class ConvolutionalAutoEncoder(nn.Module):

    def __init__(self, in_channels = 1, channels_bottleneck = 4):
        super().__init__()
        self.bottleneck = channels_bottleneck
        self.in_channels = in_channels

        self.encoder_conv = nn.Sequential(

            nn.Conv2d(in_channels=self.in_channels, out_channels=8, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(in_channels=16, out_channels=self.bottleneck, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.bottleneck),
            nn.ReLU(),

        )

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(in_channels = self.bottleneck, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels = 16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels = 8, out_channels=self.in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh(),

        )


    def forwrad_enc(self, x):
        return self.encoder_conv(x)


    def forward_dec(self, x):
        return self.decoder_conv(x)


    def forward(self, x):
        batch_size, num_channels, height, width = x.shape
        enc = self.forwrad_enc(x)
        dec = self.forward_dec(enc)

        return enc, dec



In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
# stuff to visualize the latent space
!pip install celluloid
from celluloid import Camera
from IPython.display import HTML

In [ ]:
# seed everything
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)

In [ ]:
# prep dataset

tensor_transforms = transforms.Compose(
    [
        transforms.Resize((32, 32)),
        transforms.ToTensor()
    ]
)

train_set = MNIST("/data/mnist", train=True, download=True, transform=tensor_transforms)
test_set = MNIST("/data/mnist", train=False, download=True, transform=tensor_transforms)

In [ ]:
# set device
device = "cuda" if torch.cuda.is_available() else "cpu"



building a simple linear variational autoencoder

In [ ]:
class LinearVariationalAutoEncoder(nn.Module):

    def __init__(self, latent_dim=2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
        )

        self.fn_mu = nn.Linear(32, latent_dim)
        self.fn_logvar = nn.Linear(32, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32*32),
            nn.Sigmoid(),
        )


    def forward_enc(self, x):
        x = self.encoder(x)

        mu = self.fn_mu(x)
        logvar = self.fn_logvar(x)

        sigma = torch.exp(0.5 * logvar)
        noise = torch.randn_like(sigma, device=sigma.device)

        z = mu + sigma * noise

        return z, mu, logvar


    def forward_dec(self, x):
        return self.decoder(x)


    def forward(self, x):

        batch_size, channels, height, width = x.shape

        x = x.flatten(1)

        z, mu, logvar = self.forward_enc(x)

        dec = self.forward_dec(z)

        dec = dec.reshape(batch_size, channels, height, width)

        return z, dec, mu, logvar

In [ ]:
model = LinearVariationalAutoEncoder()
rand = torch.randn(4, 1, 32, 32)


define the loss function

In [ ]:
import torch

In [ ]:
def VAELoss(x, x_hat, mean, log_var, kl_weight=1, reconstruction_weight=1):

    pixel_mse = ((x - x_hat)**2)

    pixel_mse = pixel_mse.flatten(1)

    reconstruction_loss = pixel_mse.sum(axis=-1).mean()

    kl = (1 + log_var - mean**2 - torch.exp(log_var))
    kl_per_image = -0.5 * torch.sum(kl, dim=-1)
    kl_loss = torch.mean(kl_per_image)

    return reconstruction_loss * reconstruction_weight + kl_loss * kl_weight


In [ ]:
def train(model,
          kl_weight,
          train_set,
          test_set,
          batch_size,
          training_iterations,
          evaluation_iterations,
          model_type="VAE"):

    if model_type != "VAE": kl_weight = None

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)

    trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=8)

    optimizer = optim.Adam(model.parameters(), lr=0.005)

    train_loss = []
    evaluation_loss = []

    encoded_data_per_eval = []
    train_losses = []
    evaluation_losses = []

    pbar = tqdm(range(training_iterations))

    train = True

    step_counter = 0
    while train:

        for images, labels in trainloader:

            images = images.to(device)

            if model_type == "VAE":
                encoded, decoded, mu, logvar = model(images)
                loss = VAELoss(images, decoded, mu, logvar, kl_weight)

            elif model_type == "AE":
                encoded, decoded = model(images)
                loss = torch.mean((images - decoded)**2)

            train_loss.append(loss.item())

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if step_counter % evaluation_iterations == 0:

                model.eval()

                encoded_evaluations = []

                for images, labels in testloader:
                    images = images.to(device)

                    if model_type == "VAE":
                        encoded, decoded, mu, logvar = model(images)
                        loss = VAELoss(images, decoded, mu, logvar, kl_weight)

                    elif model_type == "AE":
                        encoded, decoded = model(images)
                        loss = torch.mean((images - decoded) ** 2)

                    evaluation_loss.append(loss.item())

                    encoded, labels = encoded.cpu().flatten(1), labels.reshape(-1, 1)

                    encoded_evaluations.append(torch.cat((encoded, labels), axis=-1))

                encoded_data_per_eval.append(torch.concatenate(encoded_evaluations).detach())

                train_loss = np.mean(train_loss)
                evaluation_loss = np.mean(evaluation_loss)

                train_losses.append(train_loss)
                evaluation_losses.append(evaluation_loss)

                train_loss = []
                evaluation_loss = []

                model.train()

            step_counter += 1
            pbar.update(1)


            if step_counter >= training_iterations:
                print("completed training!")
                train = False
                break


    encoded_data_per_eval = [np.array(i) for i in encoded_data_per_eval]

    print("final training loss", train_losses[-1])
    print("final evaluation loss", evaluation_losses[-1])

    return model, train_losses, evaluation_losses, encoded_data_per_eval



In [ ]:
models = []
encoded_datas = []

# train our vanilla autoencoder
model = VanillaAutoEncoder()
model, train_losses, evaluation_losses, encoded_data_per_eval = train(model,
                                                                      kl_weight=None,
                                                                      train_set=train_set,
                                                                      test_set=test_set,
                                                                      batch_size=64,
                                                                      training_iterations=25000,
                                                                      evaluation_iterations=250,
                                                                      model_type="AE")


models.append(model)
encoded_datas.append(encoded_data_per_eval)


# train our variational autoencoders with different kl weights
kl_weights = [1, 100]
for kl_weight in kl_weights:
    print(f'kl weihght:', kl_weight)
    model = LinearVariationalAutoEncoder()
    model, train_losses, evaluation_losses, encoded_data_per_eval = train(model,
                                                                          kl_weight=kl_weight,
                                                                          train_set=train_set,
                                                                          test_set=test_set,
                                                                          batch_size=64,
                                                                          training_iterations=25000,
                                                                          evaluation_iterations=250,
                                                                          model_type="VAE")


    models.append(model)
    encoded_datas.append(encoded_data_per_eval)

comparing latents

In [ ]:
model_latents = [x[-1] for x in encoded_datas]
model_names = ["vanilla autoencoder", "vae kl wight: 1", "vae kl weight: 100"]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

for idx, encoding in enumerate(model_latents):
    encoding = pd.DataFrame(encoding, columns=["x", "y", "class"])
    encoding = encoding.sort_values(by="class")
    encoding["class"] = encoding["class"].astype(int).astype(str)

    for grouper, group in encoding.groupby("class"):
        ax[idx].scatter(x=group["x"], y=group["y"], label=grouper, alpha=0.8, s=5)
        ax[idx].set_title(model_names[idx])


# grab models for future visualization
vanilla_ae, vae, _ = models

generating new data

In [ ]:
z = torch.randn(1, 2).to("cuda")

with torch.no_grad():
    generated = vae.forward_dec(z)

generated = generated.cpu().reshape(32, 32).numpy()

plt.imshow(generated, cmap="gray")
plt.show()

let's build a convolutional variational autoencoder!

In [ ]:
class ConvolutionalAutoEncoder(nn.Module):

    def __init__(self, in_channels=1, channels_bottleneck=4):
        super().__init__()


        self.bottleneck = channels_bottleneck
        self.in_channels = in_channels


        self.encoder_conv = nn.Sequential(

            nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # self.conv_mu = nn.Conv2d(in_channels=32, out_channels=self.bottleneck, kernel_size=1, stride=1)
        # self.conv_logvar = nn.Conv2d(in_channels=32, out_channels=self.bottleneck, kernel_size=1, stride=1)

        self.conv_parameters = nn.Conv2d(in_channels=32, out_channels=2 * self.bottleneck, kernel_size = 1, stride = 1)



        self.decoder_conv = nn.Sequential(

            nn.ConvTranspose2d(in_channels=self.bottleneck, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),


            nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),


            nn.ConvTranspose2d(in_channels=8, out_channels=1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )


    def forward_dec(self, x):
        return self.decoder_conv(x)


    def forward_enc(self, x):

        x = self.encoder_conv(x)

        x = self.conv_parameters(x)

        mu, logvar = torch.chunk(x, chunks=2, dim=1)


        # mu = self.conv_mu(x)
        # logvar = self.conv_logvar(x)

        sigma = torch.exp(0.5 * logvar)
        noise = torch.randn_like(sigma)

        z = mu + sigma * noise

        return z, mu, logvar


    def forward(self, x):

        z, mu, logvar = self.forward_enc(x)

        dec = self.forward_dec(z)

        return z, dec, mu, logvar